In [1]:
import sys
from pathlib import Path

# Find the repository root by locating the src directory.
PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import numpy as np

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

#TABLE = "data/historic_airphoto_index/main_index_blank.gpkg"
#OUTPUT_DIR = Path("data/historic_airphoto_index/subsets")

#gdf = gpd.read_file(TABLE)

## Prepare the Subset

In [2]:
# CONFIGURATION CELL - EDIT HERE

# specify the county
COUNTY = "Carver"

### Examine available years for a county

In [3]:
mask = gdf["county"] == COUNTY

subset = gdf.loc[mask]

for year in sorted(subset["year"].dropna().unique()):
    print(year)

1937
1938
1940
1947
1951
1953
1955
1956
1957
1960
1962
1963
1964
1967
1969
1971


### Choose Years and Manually Populate Scale Info
Use scale information from PDFs found here: 
https://www.lib.umn.edu/collections/borchert/aerial-photography-and-remote-sensed-imagery

This notebook assumes you're going to choose a set of years that all have the same scale, for efficiency. 

In [4]:
# CONFIGURATION CELL - EDIT HERE

# Choose years for processing
# These should all have the same scale
YEARS = [1937, 1938, 1940, 1951, 1957, 1963]

SCALE = 20000

In [5]:
mask = (
    (gdf["county"] == COUNTY) &
    (gdf["year"].isin(YEARS))
)

subset_gdf = gdf.loc[mask].copy()

print(f"Matching records: {len(subset_gdf):,}")

Matching records: 871


In [6]:
missing_scale = subset_gdf["scale"].isna()

subset_gdf.loc[missing_scale, "scale"] = SCALE

print(f"Scales populated: {missing_scale.sum():,}")
print(subset_gdf["scale"].value_counts(dropna=False))

Scales populated: 871
scale
20000    871
Name: count, dtype: int64


In [7]:
# check for unexpected scale values
assert subset_gdf["scale"].eq(SCALE).all(), (
    "Some records have a different scale than expected!"
)

## Redoing main index table

In [3]:
subset_gdf = gpd.read_file("C:/Users/eu01251115/LocalProjects/batch_georef/data/main_index_blank_old.gpkg")

In [4]:

# Rename the old extent field, if present.
if "h_extent" in subset_gdf.columns:
    subset_gdf = subset_gdf.rename(
        columns={"h_extent": "wm_width"}
    )

# Numeric fields populated during photo preparation.
float_columns = [
    "dpi_x",
    "dpi_y",
    "ground_width",
    "ground_height",
    "wm_width",
    "wm_height",
]

# Pixel dimensions are whole numbers.
integer_columns = [
    "width_px",
    "height_px",
]

# Create missing fields and ensure correct types.
for col in float_columns:
    if col not in subset_gdf.columns:
        subset_gdf[col] = np.nan
    else:
        subset_gdf[col] = pd.to_numeric(
            subset_gdf[col], errors="coerce"
        )

for col in integer_columns:
    if col not in subset_gdf.columns:
        subset_gdf[col] = pd.Series(
            pd.NA,
            index=subset_gdf.index,
            dtype="Int64",
        )
    else:
        subset_gdf[col] = pd.to_numeric(
            subset_gdf[col], errors="coerce"
        ).astype("Int64")

print(subset_gdf.dtypes)

cty_year_id            str
status              object
county                 str
year                 int64
photo_id               str
wm_width           float64
ground_width       float64
gcp_count           object
confidence          object
scale               object
dpi_x              float64
width_px             Int64
worldfile_json      object
url                    str
flight_code            str
source_id            int64
date_prepared       object
date_georef         object
error               object
latitude           float64
geometry          geometry
dpi_y              float64
ground_height      float64
wm_height          float64
height_px            Int64
dtype: object


In [5]:
subset_gdf = subset_gdf[['cty_year_id','status','county','year','photo_id','ground_width','ground_height','wm_width','wm_height','scale','dpi_x','dpi_y','width_px','height_px','worldfile_json','url','flight_code','source_id','date_prepared','date_georef','gcp_count','confidence','error','latitude','geometry']]

In [6]:
subset_gdf

,cty_year_id,status,county,year,photo_id,ground_width,ground_height,wm_width,wm_height,scale,dpi_x,dpi_y,width_px,height_px,worldfile_json,url,flight_code,source_id,date_prepared,date_georef,gcp_count,confidence,error,latitude,geometry
0,aitkin_1939_bxv02017,None,Aitkin,1939,bxv02017,NaN,NaN,NaN,NaN,None,NaN,NaN,<NA>,<NA>,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxv/y1939/bxv02017.jpg,bxv,17233,None,None,None,None,None,46.164398,POINT (-10383436.935 5806733.399)
1,aitkin_1939_bxt03191,None,Aitkin,1939,bxt03191,NaN,NaN,NaN,NaN,None,NaN,NaN,<NA>,<NA>,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxt/y1939/bxt03191.jpg,bxt,17326,None,None,None,None,None,46.256401,POINT (-10442747.814 5821533.312)
2,aitkin_1939_bxt03192,None,Aitkin,1939,bxt03192,NaN,NaN,NaN,NaN,None,NaN,NaN,<NA>,<NA>,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxt/y1939/bxt03192.jpg,bxt,17327,None,None,None,None,None,46.256302,POINT (-10440599.092 5821517.341)
3,aitkin_1939_byd03058,None,Aitkin,1939,byd03058,NaN,NaN,NaN,NaN,None,NaN,NaN,<NA>,<NA>,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/byd/y1939/byd03058.jpg,byd,17348,None,None,None,None,None,46.204399,POINT (-10359703.23 5813165.04)
4,aitkin_1939_byd04003,None,Aitkin,1939,byd04003,NaN,NaN,NaN,NaN,None,NaN,NaN,<NA>,<NA>,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/byd/y1939/byd04003.jpg,byd,17350,None,None,None,None,None,46.234600,POINT (-10358746.071 5818024.084)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115572,yellow_medicine_1991_napp_4112_148,None,Yellow Medicine,1991,napp_4112_148,NaN,NaN,NaN,NaN,None,NaN,NaN,<NA>,<NA>,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4112_148.jpg,napp,9647,None,None,None,None,None,44.846100,POINT (-10655011.753 5597325.517)
115573,yellow_medicine_1991_napp_4112_149,None,Yellow Medicine,1991,napp_4112_149,NaN,NaN,NaN,NaN,None,NaN,NaN,<NA>,<NA>,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4112_149.jpg,napp,9648,None,None,None,None,None,44.877499,POINT (-10655123.006 5602256.72)
115574,yellow_medicine_1991_napp_4113_036,None,Yellow Medicine,1991,napp_4113_036,NaN,NaN,NaN,NaN,None,NaN,NaN,<NA>,<NA>,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4113_036.jpg,napp,9649,None,None,None,None,None,44.874000,POINT (-10648610.57 5601707.206)
115575,yellow_medicine_1991_napp_4113_037,None,Yellow Medicine,1991,napp_4113_037,NaN,NaN,NaN,NaN,None,NaN,NaN,<NA>,<NA>,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4113_037.jpg,napp,9650,None,None,None,None,None,44.843101,POINT (-10648410.986 5596854.756)


In [7]:
subset_gdf.columns

Index(['cty_year_id', 'status', 'county', 'year', 'photo_id', 'ground_width',
       'ground_height', 'wm_width', 'wm_height', 'scale', 'dpi_x', 'dpi_y',
       'width_px', 'height_px', 'worldfile_json', 'url', 'flight_code',
       'source_id', 'date_prepared', 'date_georef', 'gcp_count', 'confidence',
       'error', 'latitude', 'geometry'],
      dtype='str')

In [8]:
# Save subset
output_path = "C:/Users/eu01251115/LocalProjects/batch_georef/data/main_index_blank.gpkg"
subset_gdf.to_file(output_path, layer="photos", driver="GPKG")

### Save a subset index table with a fancy name

In [14]:
# Build filename components
county_name = COUNTY.strip().lower().replace(" ", "_")

year_suffix = "_".join(
    f"{year % 100:02d}" for year in sorted(set(YEARS))
)

filename = f"subset_{SCALE}_{county_name}_{year_suffix}.gpkg"

# Create output directory if needed
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / filename

# Prevent accidental overwrites
if output_path.exists():
    raise FileExistsError(f"File already exists: {output_path}")

# Save subset
subset_gdf.to_file(output_path, layer="photos", driver="GPKG")

print(f"Saved {len(subset_gdf):,} records to:")
print(output_path)

Saved 871 records to:
data\historic_airphoto_index\subsets\subset_20000_carver_37_38_40_51_57_63.gpkg


In [15]:
subset_gdf.columns

Index(['cty_year_id', 'status', 'county', 'year', 'photo_id', 'ground_width',
       'ground_height', 'wm_width', 'wm_height', 'scale', 'dpi_x', 'dpi_y',
       'width_px', 'height_px', 'worldfile_json', 'url', 'flight_code',
       'source_id', 'date_prepared', 'date_georef', 'gcp_count', 'confidence',
       'error', 'latitude', 'geometry'],
      dtype='str')

### temporary - derive carver 1963 subset

In [9]:
gdf = gpd.read_file(
    "data/historic_airphoto_index/subsets/subset_20000_carver_37_38_40_51_57_63_tracker.gpkg", 
    layer="photos", 
    driver="GPKG"
    )

gdf.head()

DataSourceError: data/historic_airphoto_index/subsets/subset_20000_carver_37_38_40_51_57_63_tracker.gpkg: No such file or directory